# BCBERT + BillingDataset + BillingTokenizer Tutorial

This notebook shows an end-to-end workflow to train and use `BCBERTForMaskedLM` with:
- `BillingTokenizer`
- `BillingDataset`
- MIMIC billing-code data (`data/mimic_4_icd_demo_with_gaps.parquet`)

You will learn how to:
1. Load and prepare data
2. Build/train a billing-code tokenizer
3. Create a `BillingDataset`
4. Train `BCBERTForMaskedLM` with masked language modeling (MLM)
5. Save and reload the trained model
6. Use the trained BCBERT for inference

## 1. Imports and setup

In [1]:
import pathlib
import sys
import random

import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
)

repo_root = pathlib.Path.cwd().parent
sys.path.insert(0, str(repo_root / "src"))

from models import (
    BillingTokenizer,
    BillingDataset,
    BCBERTConfig,
    BCBERTForMaskedLM,
    CSVLogCallback,
)

seed = 42
random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Repo root:", repo_root)

Device: cuda
Repo root: c:\Users\user\Documents\daniyal\biling-codes


## 2. Load billing data

This tutorial uses `mimic_4_icd_demo_with_gaps.parquet` so temporal gap tokens are also part of the token space.

In [23]:
data_path = repo_root / "data" / "mimic_4_icd_demo_with_gaps.parquet"
df = pd.read_parquet(data_path)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("Unique admissions:", df["hadm_id"].nunique())
print("Unique mapped codes:", df["icd_code_mapped"].nunique())

df[["subject_id", "hadm_id", "chartdate", "icd_code_mapped", "segment", "age_month"]].head(100)

Shape: (2639, 14)
Columns: ['subject_id', 'hadm_id', 'seq_num', 'chartdate', 'icd_code', 'icd_version', 'icd_code_mapped', 'pos', 'age_month', 'age_day', 'segment', 'Lo7', 'expired', 'expired_7']
Unique admissions: 500
Unique mapped codes: 763


,subject_id,hadm_id,chartdate,icd_code_mapped,segment,age_month
0,14546051,20000069,2131-11-10,0KQM0ZZ,1,400
1,14546051,20000069,2131-11-10,10E0XZZ,1,400
2,13074106,20000102,2135-10-25,M2060,1,229
3,13074106,20000102,2135-10-25,10907ZC,1,229
4,14990224,20000147,2121-08-30,02100Z9,1,872
...,...,...,...,...,...,...
95,10117812,20001770,2117-01-26,02HV33Z,1,426
96,10117812,20001770,2117-01-27,3E0G76Z,2,426
97,10117812,20001770,2117-01-28,GAP_1,2,426
98,10117812,20001770,2117-01-29,GAP_1,2,426


## 3. Split admissions and train BillingTokenizer

This follows the script flow: split by `hadm_id`, train tokenizer on train admissions only, then reuse it for train/eval datasets.

Note: in this repository, `train_from_list` works best with a flat list of codes, so we train on `train_df['icd_code_mapped']`.

In [3]:
# Optional runtime cap for notebook speed
max_admissions = 2000
hadm_ids_all = df["hadm_id"].drop_duplicates().head(max_admissions)
df_work = df[df["hadm_id"].isin(hadm_ids_all)].copy()

# Split IDs (like the training script)
hadm_ids = df_work["hadm_id"].unique()
train_ids, test_ids = train_test_split(hadm_ids, test_size=0.2, random_state=42)

train_df = df_work[df_work["hadm_id"].isin(train_ids)].copy()
test_df = df_work[df_work["hadm_id"].isin(test_ids)].copy()

print("Train admissions:", len(train_ids))
print("Test admissions:", len(test_ids))

# Train tokenizer on training set only
train_codes = train_df["icd_code_mapped"].dropna().astype(str).tolist()

tokenizer = BillingTokenizer(
    add_cls=False,
    age_ids=False,
    segment_ids=False,
)

tokenizer.train_from_list(train_codes)
tokenizer.model_max_length = 120

print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Special token ids:")
print("pad:", tokenizer.pad_token_id, "unk:", tokenizer.unk_token_id, "mask:", tokenizer.mask_token_id)

Train admissions: 400
Test admissions: 100
Vocabulary trained. Size: 677
Tokenizer vocab size: 677
Special token ids:
pad: 0 unk: 1 mask: 4


## 4. Build train/eval BillingDataset

This mirrors the script by creating admission-level grouped data for both train and eval splits.

In [4]:
train_groups = train_df.sort_values(["hadm_id", "chartdate", "seq_num"]).groupby("hadm_id")
test_groups = test_df.sort_values(["hadm_id", "chartdate", "seq_num"]).groupby("hadm_id")

# Convert to list-of-dicts as done in the script
train_data_list = train_groups.apply(lambda x: x.to_dict(orient="list")).tolist()
test_data_list = test_groups.apply(lambda x: x.to_dict(orient="list")).tolist()

train_dataset = BillingDataset(data=train_data_list, tokenizer=tokenizer, age_column="age_month")
eval_dataset = BillingDataset(data=test_data_list, tokenizer=tokenizer, age_column="age_month")

print("Train dataset size:", len(train_dataset))
print("Eval dataset size:", len(eval_dataset))

sample = train_dataset[0]
print("Sample keys:", list(sample.keys()))
for k, v in sample.items():
    shape = tuple(v.shape) if hasattr(v, "shape") else type(v)
    print(f"{k}: {shape}")

Train dataset size: 400
Eval dataset size: 100
Sample keys: ['input_ids', 'token_type_ids', 'attention_mask']
input_ids: (120,)
token_type_ids: (122,)
attention_mask: (120,)


C:\Users\user\AppData\Local\Temp\ipykernel_113364\1136492623.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_data_list = train_groups.apply(lambda x: x.to_dict(orient="list")).tolist()
C:\Users\user\AppData\Local\Temp\ipykernel_113364\1136492623.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_data_list = test_groups.apply(lambda x: x.to_dict(orient="list")).tolist()


## 5. Create BCBERTForMaskedLM (script-style config)



In [5]:

config = BCBERTConfig(
    vocab_size=tokenizer.vocab_size,

    hidden_size=64,
    num_hidden_layers=6,
    num_attention_heads=4,
    intermediate_size=128,
    max_position_embeddings=120,
    type_vocab_size=2,
    cls_pooler=False,
    enable_age_ids=False,
    enable_segment_ids=False,
)

model = BCBERTForMaskedLM(config)

print("Model created.")
print("hidden_size:", config.hidden_size)
print("num_hidden_layers:", config.num_hidden_layers)
print("num_attention_heads:", config.num_attention_heads)

Model created.
hidden_size: 64
num_hidden_layers: 6
num_attention_heads: 4


## 6. Define DataCollatorForLanguageModeling

We use Hugging Face `DataCollatorForLanguageModeling` for MLM.

In [7]:
mlm_probability = 0.4

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=mlm_probability,
)

sample_batch = [train_dataset[i] for i in range(4)]
collated = data_collator(sample_batch)

print("Collated keys:", list(collated.keys()))
for k, v in collated.items():
    print(k, tuple(v.shape))

Collated keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
input_ids (4, 120)
token_type_ids (4, 122)
attention_mask (4, 120)
labels (4, 120)


## 7. Train with Trainer + TrainingArguments

This section mirrors the script's training pipeline including early stopping and CSV logging.

In [9]:
run_name = "bcbert_gaps_mlm_tutorial"
output_dir = repo_root / "notebooks" / run_name

In [11]:

logging_dir = output_dir / "logs"
output_dir.mkdir(parents=True, exist_ok=True)
logging_dir.mkdir(parents=True, exist_ok=True)

# Optional resume behavior (script-style)
if any(output_dir.iterdir()):
    try:
        print(f"Loading existing model from {output_dir} ...")
        model = BCBERTForMaskedLM.from_pretrained(str(output_dir), config=config)
    except Exception as e:
        print("Could not load existing checkpoint; training from scratch.")
        print("Reason:", e)
        model = BCBERTForMaskedLM(config)
else:
    print("No existing model checkpoint found; training from scratch.")
    model = BCBERTForMaskedLM(config)

training_args = TrainingArguments(
    output_dir=str(output_dir),
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=3,
    logging_dir=str(logging_dir),
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=5),
        CSVLogCallback(output_dir=str(output_dir), name=run_name),
    ],
)



Loading existing model from c:\Users\user\Documents\daniyal\biling-codes\notebooks\bcbert_gaps_mlm_tutorial ...


In [ ]:
print("Starting training...")
train_result = trainer.train()
print("Training completed.")
print(train_result)

## 8. Save best model and tokenizer

In [12]:
best_model_dir = output_dir

trainer.save_model(str(best_model_dir))
tokenizer.save_pretrained(str(best_model_dir))

print(f"Best model saved to: {best_model_dir}")
print("Saved checkpoint files:")
for p in sorted(best_model_dir.iterdir()):
    print(" -", p.name)

Best model saved to: c:\Users\user\Documents\daniyal\biling-codes\notebooks\bcbert_gaps_mlm_tutorial
Saved checkpoint files:
 - checkpoint-13
 - checkpoint-26
 - config.json
 - eval_log.csv
 - logs
 - model.safetensors
 - special_tokens_map.json
 - tokenizer_config.json
 - train_log.csv
 - training_args.bin
 - vocab.json


## 9. Load trained BCBERT and tokenizer

In [13]:
trained_tokenizer = BillingTokenizer.from_pretrained(str(best_model_dir))
trained_model = BCBERTForMaskedLM.from_pretrained(str(best_model_dir), config=config).to(device)
trained_model.eval()

print("Loaded tokenizer vocab:", trained_tokenizer.vocab_size)
print("Loaded model vocab:", trained_model.config.vocab_size)
print("Loaded from:", best_model_dir)

Loaded tokenizer vocab: 677
Loaded model vocab: 677
Loaded from: c:\Users\user\Documents\daniyal\biling-codes\notebooks\bcbert_gaps_mlm_tutorial


## 10. Use trained BCBERT 

`BCBERTForMaskedLM.bert` is the underlying encoder. Its `forward()` returns:
- `last_hidden_state` — token-level representations `[batch, seq_len, hidden_size]`
- `pooler_output` — admission-level representation `[batch, hidden_size]`

Both are useful for downstream tasks such as admission-level classification or retrieval.

In [14]:
encoder = trained_model.bert  # BCBERT base encoder (no MLM head)
encoder.eval()

# Build a small batch from eval set
n_samples = 8
batch_items = [eval_dataset[i] for i in range(n_samples)]

input_ids = torch.stack([it["input_ids"] for it in batch_items]).to(device)
attention_mask = torch.stack([it["attention_mask"] for it in batch_items]).to(device)

with torch.no_grad():
    outputs = encoder(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )

# token-level representations: [batch, seq_len, hidden_size]
token_embeddings = outputs.last_hidden_state

# admission-level (pooled) embeddings: [batch, hidden_size]
# mean-pooled over non-PAD tokens (cls_pooler=False default)
admission_embeddings = outputs.pooler_output

print("token_embeddings shape:", tuple(token_embeddings.shape))
print("admission_embeddings shape:", tuple(admission_embeddings.shape))

# Inspect one embedding vector
print("\nSample admission embedding (first 10 dims):")
print(admission_embeddings[0, :10].cpu().numpy().round(4))

# Cosine similarity matrix between all admission embeddings
from torch.nn.functional import normalize

normed = normalize(admission_embeddings, dim=-1)
sim_matrix = (normed @ normed.T).cpu().numpy().round(3)

import pandas as pd
hadm_labels = [f"adm_{i}" for i in range(n_samples)]
print("\nCosine similarity matrix:")
pd.DataFrame(sim_matrix, index=hadm_labels, columns=hadm_labels)

token_embeddings shape: (8, 120, 64)
admission_embeddings shape: (8, 64)

Sample admission embedding (first 10 dims):
[-0.5983  0.4014  0.8419 -1.5694 -0.6765  0.3963 -0.67   -0.3241 -1.03
  0.3796]

Cosine similarity matrix:


,adm_0,adm_1,adm_2,adm_3,adm_4,adm_5,adm_6,adm_7
adm_0,1.000,0.787,0.507,0.683,0.741,0.786,0.809,0.603
adm_1,0.787,1.000,0.620,0.751,0.841,0.817,0.775,0.695
adm_2,0.507,0.620,1.000,0.865,0.725,0.560,0.531,0.924
adm_3,0.683,0.751,0.865,1.000,0.773,0.833,0.779,0.865
adm_4,0.741,0.841,0.725,0.773,1.000,0.751,0.744,0.780
adm_5,0.786,0.817,0.560,0.833,0.751,1.000,0.868,0.636
adm_6,0.809,0.775,0.531,0.779,0.744,0.868,1.000,0.648
adm_7,0.603,0.695,0.924,0.865,0.780,0.636,0.648,1.000


In [15]:
admission_embeddings[0]

tensor([-0.5983,  0.4014,  0.8419, -1.5694, -0.6765,  0.3963, -0.6700, -0.3241,
        -1.0300,  0.3796, -0.1749,  1.2637, -0.0828,  0.3974,  0.1739, -0.1793,
         0.3278, -0.2029,  0.5452,  0.8606, -0.6997,  1.0583, -1.0892, -0.1729,
         0.4050, -1.3502,  0.5564,  0.2041,  0.8314, -0.5227, -0.6712, -1.0775,
         1.2633,  0.9087, -0.0469,  1.0433, -0.3312,  0.9581,  0.3669,  0.9488,
         0.0118, -1.0359, -0.8921, -0.2700,  0.2086,  1.0262,  0.6026, -1.1702,
        -0.8420, -2.2383, -0.2180,  1.1932, -1.1225,  0.8282,  1.2888,  0.4911,
         0.0114,  0.3520, -0.1528,  0.8906, -1.2757, -0.3970,  0.4825, -0.4461],
       device='cuda:0')

You now have a script-style BCBERT workflow:
- split admissions into train/eval sets
- train `BillingTokenizer` on train data
- build `BillingDataset` objects for Trainer
- train `BCBERTForMaskedLM` with `Trainer` + `TrainingArguments`
- use `DataCollatorForLanguageModeling` for MLM masking
- save and reload best model/tokenizer
- run masked-token inference with the trained model